# 라이브러리 및 데이터 불러오기

In [1]:
import pandas as pd
import numpy as np

In [2]:
from google.cloud import bigquery

PROJECT_ID = "sns-analysis-prj"
DATA_SET = "sns_analysis"

client = bigquery.Client(project=PROJECT_ID)

In [ ]:
sql = f"""
    SELECT user_id, COUNT(user_id) AS cnt
    FROM `{PROJECT_ID}.{DATA_SET}.user_friend_count_history`
    GROUP BY user_id
"""

# 판다스 데이터프레임으로 변환
df = client.query(sql).to_dataframe()

df.head()

,user_id,cnt
0,1216453,86
1,1244774,79
2,897734,147
3,1269996,77
4,971175,90


In [4]:
df['cnt'].median()

np.float64(35.0)

In [ ]:
sql2 = f"""
    SELECT user_id, COUNT(user_id) AS cnt
    FROM `{PROJECT_ID}.{DATA_SET}.user_friend_count_history` AS h
    INNER JOIN `{PROJECT_ID}.{DATA_SET}.accounts_user` AS u
    ON h.user_id = u.id
    INNER JOIN `{PROJECT_ID}.{DATA_SET}.accounts_group` AS g
    ON u.group_id = g.id
    INNER JOIN `{PROJECT_ID}.{DATA_SET}.accounts_group` AS s 
    ON g.school_id = s.id
    WHERE g.school_id IN (271, 352, 369, 1478, 1719, 4426, 4516, 5372, 5491, 5520)
    GROUP BY h.user_id
"""

# 판다스 데이터프레임으로 변환
df2 = client.query(sql2).to_dataframe()

df2.head()

,user_id,cnt
0,1238616,34
1,866250,28
2,1142318,130
3,890065,12
4,889999,15


In [6]:
df2['cnt'].median()

np.float64(39.0)

In [8]:
first_friend_sql = f"""
WITH first_friendship AS (
    SELECT
        user_id,
        MIN(friendship_at) AS first_friendship_at
    FROM `{PROJECT_ID}.{DATA_SET}.user_friend_count_history`
    GROUP BY user_id
)

SELECT
    u.id AS user_id,
    u.created_at AS signup_at,
    f.first_friendship_at,

    TIMESTAMP_DIFF(
        f.first_friendship_at,
        u.created_at,
        SECOND
    ) / 3600.0 AS hours_to_first_friendship,

    TIMESTAMP_DIFF(
        f.first_friendship_at,
        u.created_at,
        SECOND
    ) / 86400.0 AS days_to_first_friendship

FROM `{PROJECT_ID}.{DATA_SET}.accounts_user` AS u
INNER JOIN first_friendship AS f
    ON u.id = f.user_id
"""

first_friend_timing = (
    client.query(first_friend_sql)
    .to_dataframe()
)

display(first_friend_timing.head())
display(first_friend_timing['hours_to_first_friendship'].median())

,user_id,signup_at,first_friendship_at,hours_to_first_friendship,days_to_first_friendship
0,837754,2023-04-19 06:54:56.555770+00:00,2023-04-19 06:55:15+00:00,0.005000,0.000208
1,838998,2023-04-20 07:19:56.721909+00:00,2023-04-20 08:24:04+00:00,1.068611,0.044525
2,1205856,2023-05-13 08:47:45.857697+00:00,2023-05-13 08:59:32+00:00,0.196111,0.008171
3,860630,2023-04-30 14:22:33.142870+00:00,2023-04-30 14:57:34+00:00,0.583333,0.024306
4,861153,2023-04-30 15:03:46.223145+00:00,2023-04-30 15:27:15+00:00,0.391111,0.016296


np.float64(0.18138888888888888)

In [ ]:
first_friend_sql = f"""
WITH first_friendship AS (
    SELECT
        user_id,
        MIN(friendship_at) AS first_friendship_at
    FROM `{PROJECT_ID}.{DATA_SET}.user_friend_count_history`
    GROUP BY user_id
)

SELECT
    u.id AS user_id,
    u.created_at AS signup_at,
    f.first_friendship_at,

    TIMESTAMP_DIFF(
        f.first_friendship_at,
        u.created_at,
        SECOND
    ) / 3600.0 AS hours_to_first_friendship,

    TIMESTAMP_DIFF(
        f.first_friendship_at,
        u.created_at,
        SECOND
    ) / 86400.0 AS days_to_first_friendship

FROM `{PROJECT_ID}.{DATA_SET}.accounts_user` AS u
INNER JOIN first_friendship AS f
    ON u.id = f.user_id
"""

first_friend_timing = (
    client.query(first_friend_sql)
    .to_dataframe()
)

display(first_friend_timing.head())
display(first_friend_timing['hours_to_first_friendship'].median())

In [9]:
target_school_ids = (
    271, 352, 369, 1478, 1719,
    4426, 4516, 5372, 5491, 5520,
)

In [10]:
first_friend_sql = f"""
WITH target_users AS (
    SELECT
        u.id AS user_id,
        u.created_at AS signup_at,
        u.group_id,
        g.school_id
    FROM `{PROJECT_ID}.{DATA_SET}.accounts_user` AS u
    INNER JOIN `{PROJECT_ID}.{DATA_SET}.accounts_group` AS g
        ON u.group_id = g.id
    WHERE g.school_id IN (
        271, 352, 369, 1478, 1719,
        4426, 4516, 5372, 5491, 5520
    )
),

first_friendship AS (
    SELECT
        user_id,
        MIN(friendship_at) AS first_friendship_at
    FROM `{PROJECT_ID}.{DATA_SET}.user_friend_count_history`
    GROUP BY user_id
)

SELECT
    u.user_id,
    u.school_id,
    u.group_id,
    u.signup_at,
    f.first_friendship_at,

    TIMESTAMP_DIFF(
        f.first_friendship_at,
        u.signup_at,
        SECOND
    ) / 3600.0 AS hours_to_first_friendship,

    TIMESTAMP_DIFF(
        f.first_friendship_at,
        u.signup_at,
        SECOND
    ) / 86400.0 AS days_to_first_friendship

FROM target_users AS u
LEFT JOIN first_friendship AS f
    ON u.user_id = f.user_id
"""

first_friend_timing = (
    client.query(first_friend_sql)
    .to_dataframe()
)

display(first_friend_timing.head())

,user_id,school_id,group_id,signup_at,first_friendship_at,hours_to_first_friendship,days_to_first_friendship
0,1116196,352,12343,2023-05-11 02:55:31.224481+00:00,2023-05-11 03:00:45+00:00,0.086944,0.003623
1,1007303,4516,10276,2023-05-08 12:03:56.593162+00:00,2023-05-08 12:21:20+00:00,0.289722,0.012072
2,870224,369,6513,2023-05-02 09:34:43.446143+00:00,2023-05-02 09:35:18+00:00,0.009444,0.000394
3,888666,271,4286,2023-05-05 14:58:46.649592+00:00,2023-05-05 15:59:00+00:00,1.003611,0.041817
4,884682,4516,9418,2023-05-05 08:27:51.933459+00:00,2023-05-05 08:30:39+00:00,0.046389,0.001933


In [11]:
display(first_friend_timing['hours_to_first_friendship'].median())

np.float64(0.10194444444444445)